# PoseCoach P02 — Fit3D: Steps 11b → 12 (Google Colab Edition)

> **Run on:** Google Colab | **Data:** Google Drive (`GYMVISION AI/datasets/fit3d/`)
>
> This notebook picks up exactly where the Kaggle P01 notebook left off (after Step 10c).
> It covers the Fit3D-specific pipeline that must run in Colab because the data lives in Drive.

### Steps covered:
| Step | What it does | Output |
|------|-------------|--------|
| **11b** | Explore Fit3D dataset structure | Console summary |
| **11c** | Compute Golden Angle Templates from 3D skeletons | `golden_angle_ranges.json` |
| **12**  | Validate rep counter with Fit3D ground truth | `rep_counter_validation.json` |

---
### Your Drive structure (already confirmed ✅)
```
MyDrive/GYMVISION AI/datasets/fit3d/
├── fit3d_info.json
├── raw/
│   ├── train/   s03  s04  s05  s07  s08  s09  s10  s11   (8 subjects)
│   └── test/    s02  s12  s13                             (3 subjects)
│       each subject/
│           ├── joints3d_25/   <exercise>.json   ← 3D joint positions [N,25,3]
│           ├── smplx/
│           ├── camera_parameters/
│           ├── videos/
│           └── rep_ann.json                     ← rep timing annotations
```

## 🔧 Setup — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT          = '/content/drive/MyDrive'
GYMVISION_ROOT      = f'{DRIVE_ROOT}/GYMVISION AI'
FIT3D_BASE          = f'{GYMVISION_ROOT}/datasets/fit3d'
FIT3D_RAW           = f'{FIT3D_BASE}/raw'
FIT3D_INFO_PATH     = f'{FIT3D_BASE}/fit3d_info.json'
ANGLE_TEMPLATES_DIR = f'{FIT3D_BASE}/angle_templates'
EVAL_DIR            = f'{GYMVISION_ROOT}/data/eval'

for d in [ANGLE_TEMPLATES_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

# Quick sanity check
train_subjects = sorted(os.listdir(f'{FIT3D_RAW}/train'))
test_subjects  = sorted(os.listdir(f'{FIT3D_RAW}/test'))
print(f'✅ Drive mounted')
print(f'   Train subjects ({len(train_subjects)}): {train_subjects}')
print(f'   Test  subjects ({len(test_subjects)}):  {test_subjects}')

---
## ✅ Step 11b — Explore Fit3D Dataset Structure

Maps out subjects, exercises, and file counts so you know exactly what data is available.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

fit3d_root = Path(FIT3D_RAW)

# ── 1. Load dataset info ──────────────────────────────────────────────────────
with open(FIT3D_INFO_PATH) as f:
    fit3d_info = json.load(f)
print('=== fit3d_info.json top-level keys:', list(fit3d_info.keys()))

# ── 2. Walk structure ─────────────────────────────────────────────────────────
print('\n=== Dataset Structure ===')
exercise_set = set()
subject_exercise_counts = defaultdict(lambda: defaultdict(int))  # {split: {subject: count}}

for split in ['train', 'test']:
    split_path = fit3d_root / split
    if not split_path.exists():
        print(f'  ⚠️  {split}/ not found — skipping')
        continue
    subjects = sorted(os.listdir(split_path))
    print(f'\n[{split}] {len(subjects)} subjects')

    for subj in subjects:
        subj_path = split_path / subj
        j3d_path  = subj_path / 'joints3d_25'

        if j3d_path.exists():
            exercise_files = sorted(j3d_path.glob('*.json'))
            exercises = [f.stem for f in exercise_files]
            exercise_set.update(exercises)
            subject_exercise_counts[split][subj] = len(exercises)
            print(f'  {subj:6s}  joints3d_25/  → {len(exercises):3d} exercises')
        else:
            print(f'  {subj:6s}  ⚠️  No joints3d_25/ folder found')

        # Check for rep annotations
        rep_file = subj_path / 'rep_ann.json'
        if rep_file.exists():
            with open(rep_file) as f:
                rep_data = json.load(f)
            print(f'          rep_ann.json  → {len(rep_data)} sequences annotated')

print(f'\n=== Total unique exercises across all subjects: {len(exercise_set)} ===')
for ex in sorted(exercise_set):
    print(f'  {ex}')

In [ ]:
# ── 3. Inspect one JSON to understand the joint data format ───────────────────
import numpy as np

# Pick first train subject, first exercise
sample_subj = sorted(os.listdir(f'{FIT3D_RAW}/train'))[0]
sample_dir  = Path(FIT3D_RAW) / 'train' / sample_subj / 'joints3d_25'
sample_file = sorted(sample_dir.glob('*.json'))[0]

with open(sample_file) as f:
    sample_data = json.load(f)

print(f'Sample file: {sample_file.name}')
print(f'Top-level keys: {list(sample_data.keys())}')

# Auto-detect the joints array
joints_array = None
joints_key   = None
for key, val in sample_data.items():
    if isinstance(val, list) and len(val) > 0:
        arr = np.array(val)
        if arr.ndim == 3:  # [N_frames, N_joints, 3]
            joints_array = arr
            joints_key   = key
            break

if joints_array is not None:
    print(f'\n✅ Found 3D joints under key "{joints_key}"')
    print(f'   Shape: {joints_array.shape}  → [N_frames={joints_array.shape[0]}, N_joints={joints_array.shape[1]}, xyz={joints_array.shape[2]}]')
    print(f'   Value range: [{joints_array.min():.3f}, {joints_array.max():.3f}] meters')
    print(f'   Frame 0, joint 0 (pelvis): {joints_array[0, 0]}')
else:
    print('⚠️  Could not auto-detect joints array — printing keys + shapes:')
    for k, v in sample_data.items():
        shape = np.array(v).shape if isinstance(v, list) else type(v)
        print(f'  {k}: {shape}')

# Store for later cells
JOINTS_KEY = joints_key or 'joints3d_25'

---
## ✅ Step 11c — Compute Golden Angle Templates from 3D Skeletons

### Fit3D 25-Joint Skeleton — **Limb-first ordering** (verified via z-heights)
```
 0: pelvis        1: l_hip         2: l_knee        3: l_ankle
 4: r_hip         5: r_knee        6: r_ankle
 7: spine         8: neck          9: head         10: head_top
11: l_shoulder   12: l_elbow      13: l_wrist
14: r_shoulder   15: r_elbow      16: r_wrist
17: l_foot       18: l_foot_2     19: r_foot       20: r_foot_2
21: l_hand       22: l_hand_2     23: r_hand       24: r_hand_2
```

> ✅ Ordering confirmed by diagnostic: l_hip→l_knee = 0.471 m, l_knee→l_ankle = 0.461 m.
> Legs are contiguous (0-6), NOT interleaved like SMPL-X spine-first ordering.


In [ ]:
import numpy as np

# ── Fit3D 25-joint indices — LIMB-FIRST ordering (verified) ──────────────────
# Legs are contiguous: pelvis(0), then full left leg chain, then right leg chain.
# This was confirmed diagnostically: joint[1]→[2] = 0.471m (thigh), [2]→[3] = 0.461m (shank).
JOINT_NAMES = [
    'pelvis',     # 0
    'l_hip',      # 1
    'l_knee',     # 2
    'l_ankle',    # 3
    'r_hip',      # 4
    'r_knee',     # 5
    'r_ankle',    # 6
    'spine',      # 7
    'neck',       # 8
    'head',       # 9
    'head_top',   # 10
    'l_shoulder', # 11
    'l_elbow',    # 12
    'l_wrist',    # 13
    'r_shoulder', # 14
    'r_elbow',    # 15
    'r_wrist',    # 16
    'l_foot',     # 17
    'l_foot_2',   # 18
    'r_foot',     # 19
    'r_foot_2',   # 20
    'l_hand',     # 21
    'l_hand_2',   # 22
    'r_hand',     # 23
    'r_hand_2',   # 24
]

J = {name: idx for idx, name in enumerate(JOINT_NAMES)}

# ── Angle triplets: (joint_A, vertex_joint_B, joint_C) ───────────────────────
ANGLE_TRIPLETS = {
    # Lower body
    'left_knee_angle':     (1,  2,  3),   # l_hip  → l_knee  ← l_ankle
    'right_knee_angle':    (4,  5,  6),   # r_hip  → r_knee  ← r_ankle
    'left_hip_angle':      (7,  1,  2),   # spine  → l_hip   ← l_knee
    'right_hip_angle':     (7,  4,  5),   # spine  → r_hip   ← r_knee
    'left_ankle_angle':    (2,  3, 17),   # l_knee → l_ankle ← l_foot
    'right_ankle_angle':   (5,  6, 19),   # r_knee → r_ankle ← r_foot
    # Upper body
    'left_elbow_angle':    (11, 12, 13),  # l_shoulder → l_elbow ← l_wrist
    'right_elbow_angle':   (14, 15, 16),  # r_shoulder → r_elbow ← r_wrist
    'left_shoulder_angle': (7,  11, 12),  # spine → l_shoulder ← l_elbow
    'right_shoulder_angle':(7,  14, 15),  # spine → r_shoulder ← r_elbow
    # Trunk
    'trunk_flex':          (8,   7,  0),  # neck → spine ← pelvis
    'hip_trunk_angle':     (1,   0,  4),  # l_hip → pelvis ← r_hip
}

print('✅ CORRECTED joint mapping loaded (limb-first ordering)')
print('   Angle triplets:')
for name, (a, b, c) in ANGLE_TRIPLETS.items():
    print(f'  {name:30s}  {JOINT_NAMES[a]:12s} → {JOINT_NAMES[b]:12s} ← {JOINT_NAMES[c]}')


In [ ]:
# ── Angle computation utilities ───────────────────────────────────────────────

def compute_angle_3d(a, b, c):
    """Angle (degrees) at joint b, formed by vectors b→a and b→c.
    a, b, c are 3D numpy arrays."""
    ba = a - b
    bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

def compute_angles_sequence(joints_seq, triplets):
    """Compute all angles for every frame in a sequence.
    joints_seq: np.ndarray [N_frames, 25, 3]
    Returns dict {angle_name: np.array([N_frames])}
    """
    result = {name: [] for name in triplets}
    for frame in joints_seq:
        for name, (ia, ib, ic) in triplets.items():
            result[name].append(compute_angle_3d(frame[ia], frame[ib], frame[ic]))
    return {k: np.array(v) for k, v in result.items()}

def load_joints3d(json_path):
    """Load 3D joints from a Fit3D JSON file.
    Returns np.ndarray [N_frames, 25, 3] or None on failure.
    """
    with open(json_path) as f:
        data = json.load(f)
    # Try known key first, then auto-detect
    for key in [JOINTS_KEY, 'joints3d_25', 'joints3d', 'joints']:
        if key in data:
            arr = np.array(data[key])
            if arr.ndim == 3 and arr.shape[-1] == 3:
                return arr
    # Fallback: find any 3D array
    for key, val in data.items():
        if isinstance(val, list):
            arr = np.array(val)
            if arr.ndim == 3 and arr.shape[-1] == 3:
                return arr
    return None

print('✅ Utility functions defined')

In [ ]:
# ── Compute angle distributions across ALL train subjects ─────────────────────
# This is the main loop. May take 5–15 min depending on Drive read speed.

from collections import defaultdict
import time

angle_stats = defaultdict(lambda: defaultdict(list))  # {exercise: {angle_name: [all_values]}}
errors = []

train_path = Path(FIT3D_RAW) / 'train'
train_subjects = sorted(os.listdir(train_path))

t0 = time.time()
for subj in train_subjects:
    j3d_path = train_path / subj / 'joints3d_25'
    if not j3d_path.exists():
        print(f'  ⚠️  {subj}: no joints3d_25/ folder')
        continue

    exercise_files = sorted(j3d_path.glob('*.json'))
    print(f'{subj}  ({len(exercise_files)} exercises) ...', end=' ', flush=True)

    for jf in exercise_files:
        exercise = jf.stem
        try:
            joints = load_joints3d(jf)
            if joints is None:
                errors.append(f'{subj}/{exercise}: could not parse joints')
                continue
            angles = compute_angles_sequence(joints, ANGLE_TRIPLETS)
            for ang_name, vals in angles.items():
                angle_stats[exercise][ang_name].extend(vals.tolist())
        except Exception as e:
            errors.append(f'{subj}/{exercise}: {e}')

    print('done')

elapsed = time.time() - t0
print(f'\n✅ Done in {elapsed:.0f}s')
print(f'   Exercises with angle data: {len(angle_stats)}')
if errors:
    print(f'   ⚠️  {len(errors)} errors:')
    for e in errors[:10]:
        print(f'      {e}')

In [ ]:
# ── Build ANGLE_RANGES dict and save ─────────────────────────────────────────
# For each exercise × angle: compute p5/p25/p50/p75/p95 percentiles

ANGLE_RANGES = {}

for exercise, angle_dict in sorted(angle_stats.items()):
    ANGLE_RANGES[exercise] = {}
    for angle_name, values in sorted(angle_dict.items()):
        arr = np.array(values)
        ANGLE_RANGES[exercise][angle_name] = {
            'p5':    round(float(np.percentile(arr, 5)),  1),
            'p25':   round(float(np.percentile(arr, 25)), 1),
            'p50':   round(float(np.percentile(arr, 50)), 1),
            'p75':   round(float(np.percentile(arr, 75)), 1),
            'p95':   round(float(np.percentile(arr, 95)), 1),
            'min':   round(float(arr.min()),  1),
            'max':   round(float(arr.max()),  1),
            'n_frames': len(arr),
        }

# ── Save to Drive ─────────────────────────────────────────────────────────────
out_path = f'{ANGLE_TEMPLATES_DIR}/golden_angle_ranges.json'
with open(out_path, 'w') as f:
    json.dump(ANGLE_RANGES, f, indent=2)

print(f'✅ Saved → {out_path}')
print(f'   {len(ANGLE_RANGES)} exercises covered\n')

# ── Preview a few exercises ───────────────────────────────────────────────────
PREVIEW_EXERCISES = ['squat', 'pushup', 'dumbbell_biceps_curls', 'burpees']
for ex in PREVIEW_EXERCISES:
    if ex in ANGLE_RANGES:
        print(f'--- {ex} ---')
        for angle, stats in ANGLE_RANGES[ex].items():
            if 'knee' in angle or 'elbow' in angle:
                print(f'  {angle:30s}  p5={stats["p5"]:5.1f}°  p50={stats["p50"]:5.1f}°  p95={stats["p95"]:5.1f}°  (n={stats["n_frames"]})')
        print()

---
## ✅ Step 12 — Validate Rep Counter with Fit3D Ground Truth

Each subject has a `rep_ann.json` file with the ground-truth rep count per exercise.
We simulate what `rep_counter.py` does (peak detection on angle oscillations) and measure accuracy.

**Metric:** `rep_acc = 1 - |predicted - gt| / gt`  (clipped to [0, 1])

In [ ]:
# ── Explore rep_ann.json format ───────────────────────────────────────────────
from pathlib import Path

# Load one rep_ann to understand its structure
sample_subj = sorted(os.listdir(f'{FIT3D_RAW}/train'))[0]
rep_ann_path = Path(FIT3D_RAW) / 'train' / sample_subj / 'rep_ann.json'

with open(rep_ann_path) as f:
    rep_ann = json.load(f)

print(f'rep_ann.json from {sample_subj}:')
print(f'  Top-level keys: {list(rep_ann.keys())[:10]}')

# Print first 3 entries to understand format
for ex, val in list(rep_ann.items())[:3]:
    print(f'\n  [{ex}]')
    if isinstance(val, dict):
        print(f'    keys: {list(val.keys())}')
        for k, v in val.items():
            print(f'    {k}: {v}')
    else:
        print(f'    value: {val}')

In [ ]:
from scipy.signal import find_peaks

def count_reps_from_angles(angle_sequence, prominence=10, distance=15):
    """Count reps by detecting peaks in a joint angle time series.
    prominence : minimum angle change to count as a rep (degrees)
    distance   : minimum frames between reps (at 50fps, 15 frames = 0.3s)
    Returns (rep_count, peak_indices)
    """
    if len(angle_sequence) < 10:
        return 0, []
    peaks, _ = find_peaks(angle_sequence, prominence=prominence, distance=distance)
    return len(peaks), peaks.tolist()

def get_gt_rep_count(rep_ann_entry):
    """Extract ground-truth rep count from a rep_ann.json entry.
    Fit3D format: list of peak frame indices, e.g. [97, 215, 304, 409] = 4 reps.
    """
    if isinstance(rep_ann_entry, list):
        return len(rep_ann_entry)          # ← Fit3D actual format
    if isinstance(rep_ann_entry, int):
        return rep_ann_entry
    if isinstance(rep_ann_entry, dict):
        for key in ['reps', 'rep_count', 'count', 'n_reps', 'num_reps']:
            if key in rep_ann_entry:
                return int(rep_ann_entry[key])
        for key in ['intervals', 'segments', 'boundaries']:
            if key in rep_ann_entry:
                return len(rep_ann_entry[key])
    return None

def get_best_angle_for_rep_counting(exercise, angle_ranges):
    """Return the angle name with the highest oscillation amplitude (p95-p5)."""
    if exercise not in angle_ranges:
        return 'left_knee_angle'
    ex_angles = angle_ranges[exercise]
    best = max(ex_angles.items(), key=lambda kv: kv[1]['p95'] - kv[1]['p5'])
    return best[0]

print('✅ Rep counting utilities defined')
print('   get_gt_rep_count handles: list (Fit3D native), int, dict')


In [ ]:
# ── Run validation across train subjects ─────────────────────────────────────
# For each subject × exercise: predict rep count, compare to ground truth.

validation_results = []
ANGLE_NAME_TO_IDX = {name: idx for idx, name in enumerate(JOINT_NAMES)}

for subj in sorted(os.listdir(f'{FIT3D_RAW}/train')):
    subj_path = Path(FIT3D_RAW) / 'train' / subj
    rep_file  = subj_path / 'rep_ann.json'
    j3d_path  = subj_path / 'joints3d_25'

    if not rep_file.exists() or not j3d_path.exists():
        continue

    with open(rep_file) as f:
        rep_ann = json.load(f)

    for exercise_json in sorted(j3d_path.glob('*.json')):
        exercise = exercise_json.stem
        gt_entry = rep_ann.get(exercise)
        if gt_entry is None:
            continue  # exercise not annotated

        gt_reps = get_gt_rep_count(gt_entry)
        if gt_reps is None or gt_reps == 0:
            continue

        joints = load_joints3d(exercise_json)
        if joints is None:
            continue

        # Compute angles and pick best one for rep counting
        angles = compute_angles_sequence(joints, ANGLE_TRIPLETS)
        best_angle = get_best_angle_for_rep_counting(exercise, ANGLE_RANGES)
        angle_seq  = angles.get(best_angle, angles[list(angles.keys())[0]])

        pred_reps, _ = count_reps_from_angles(angle_seq)
        acc = max(0.0, 1.0 - abs(pred_reps - gt_reps) / gt_reps)

        validation_results.append({
            'subject':    subj,
            'exercise':   exercise,
            'gt_reps':    gt_reps,
            'pred_reps':  pred_reps,
            'accuracy':   round(acc, 3),
            'angle_used': best_angle,
        })

print(f'Validated {len(validation_results)} exercise sequences')

# ── Summary stats ─────────────────────────────────────────────────────────────
if validation_results:
    accs = [r['accuracy'] for r in validation_results]
    mean_acc = np.mean(accs)
    pct_perfect = sum(1 for a in accs if a == 1.0) / len(accs) * 100
    pct_gte90   = sum(1 for a in accs if a >= 0.9)  / len(accs) * 100

    print(f'\n=== Rep Counter Validation ===')
    print(f'  Mean accuracy:      {mean_acc:.3f}  ({mean_acc*100:.1f}%)')
    print(f'  Perfect (100%):     {pct_perfect:.1f}% of sequences')
    print(f'  ≥90% accuracy:      {pct_gte90:.1f}% of sequences')

    # Per-exercise breakdown
    from collections import defaultdict
    per_ex = defaultdict(list)
    for r in validation_results:
        per_ex[r['exercise']].append(r['accuracy'])
    print('\n  Per-exercise mean accuracy (worst first):')
    for ex, ex_accs in sorted(per_ex.items(), key=lambda kv: np.mean(kv[1])):
        print(f'    {ex:40s}  {np.mean(ex_accs):.3f}  (n={len(ex_accs)})')

In [ ]:
# ── Save validation results ───────────────────────────────────────────────────
import json, os

out = {
    'summary': {
        'n_sequences':   len(validation_results),
        'mean_accuracy': round(float(np.mean([r['accuracy'] for r in validation_results])), 4) if validation_results else 0,
        'pct_perfect':   round(pct_perfect, 1) if validation_results else 0,
        'pct_gte90':     round(pct_gte90,   1) if validation_results else 0,
    },
    'results': validation_results,
}

eval_path = f'{EVAL_DIR}/rep_counter_validation.json'
with open(eval_path, 'w') as f:
    json.dump(out, f, indent=2)

print(f'✅ Saved → {eval_path}')
print()

# Also confirm golden angle ranges are saved
ranges_path = f'{ANGLE_TEMPLATES_DIR}/golden_angle_ranges.json'
sz = os.path.getsize(ranges_path) / 1024
print(f'✅ Golden angle ranges: {ranges_path}  ({sz:.0f} KB)')

---
## 🔬 Step 12b — Improved Rep Counter (Adaptive Prominence + Smoothing)

### Why 63.2% baseline fails on complex exercises
- **`dumbbell_curl_trifecta`**, **`man_maker`**, **`burpees`**: multi-movement compounds —
  a single angle oscillates at a sub-rep frequency.
- **`pushup`**, **`mule_kick`**: short-range or asymmetric peaks swamped by noise.

### Improvements
| Fix | What it does |
|---|---|
| Gaussian smoothing (σ = fps/10) | Kills high-freq noise while preserving rep peaks |
| Adaptive prominence | Set to 25 % of IQR from `golden_angle_ranges.json` |
| Peaks **and** valleys | Uses whichever has better spacing regularity |
| Multi-angle voting | Averages count from top-3 oscillating angles |


In [ ]:
from scipy.ndimage import gaussian_filter1d
import numpy as np

FPS = 50  # Fit3D capture rate

def _peak_spacing_regularity(peak_indices):
    """Return coefficient of variation of inter-peak intervals (lower = more regular)."""
    if len(peak_indices) < 2:
        return float('inf')
    gaps = np.diff(peak_indices)
    return float(np.std(gaps) / (np.mean(gaps) + 1e-8))

def count_reps_adaptive(angles_dict, exercise, angle_ranges, fps=FPS):
    """
    Improved rep counter:
      1. Gaussian-smooth each angle signal
      2. Set prominence = max(5, 0.25 * IQR from golden templates)
      3. Detect peaks AND valleys; keep the direction with better regularity
      4. Vote across the top-3 highest-amplitude angles; return the median
    """
    if not angles_dict:
        return 0, []

    # ── Rank angles by oscillation amplitude (p95-p5 from templates) ──────────
    if exercise in angle_ranges:
        ranked = sorted(
            angle_ranges[exercise].keys(),
            key=lambda a: angle_ranges[exercise][a]['p95'] - angle_ranges[exercise][a]['p5'],
            reverse=True
        )
    else:
        ranked = list(angles_dict.keys())

    # Use top-3 angles (or all if fewer)
    top_angles = [a for a in ranked if a in angles_dict][:3]

    counts = []
    best_peaks = []

    for ang_name in top_angles:
        seq = np.array(angles_dict[ang_name])
        if len(seq) < 20:
            continue

        # Smooth
        sigma = fps / 10.0   # ~5 frames at 50 fps
        smoothed = gaussian_filter1d(seq, sigma=sigma)

        # Adaptive prominence
        if exercise in angle_ranges and ang_name in angle_ranges[exercise]:
            s = angle_ranges[exercise][ang_name]
            iqr = s['p75'] - s['p25']
            prominence = max(5.0, iqr * 0.25)
        else:
            prominence = 10.0

        # Min half-period: assume ≥ 0.5 s per rep half-cycle
        distance = max(10, int(fps * 0.5))

        peaks, _   = find_peaks( smoothed, prominence=prominence, distance=distance)
        valleys, _ = find_peaks(-smoothed, prominence=prominence, distance=distance)

        # Pick direction with better periodicity regularity
        reg_p = _peak_spacing_regularity(peaks)
        reg_v = _peak_spacing_regularity(valleys)

        if reg_p <= reg_v:
            chosen, chosen_peaks = len(peaks), peaks
        else:
            chosen, chosen_peaks = len(valleys), valleys

        counts.append(chosen)
        if not best_peaks:
            best_peaks = chosen_peaks.tolist()

    if not counts:
        return 0, []
    # Median vote across top-3 angles
    rep_count = int(np.median(counts))
    return rep_count, best_peaks


# ── Re-run validation with improved counter ───────────────────────────────────
validation_improved = []

for subj in sorted(os.listdir(f'{FIT3D_RAW}/train')):
    subj_path = Path(FIT3D_RAW) / 'train' / subj
    rep_file  = subj_path / 'rep_ann.json'
    j3d_path  = subj_path / 'joints3d_25'

    if not rep_file.exists() or not j3d_path.exists():
        continue

    with open(rep_file) as f:
        rep_ann = json.load(f)

    for exercise_json in sorted(j3d_path.glob('*.json')):
        exercise = exercise_json.stem
        gt_entry = rep_ann.get(exercise)
        if gt_entry is None:
            continue
        gt_reps = get_gt_rep_count(gt_entry)
        if gt_reps is None or gt_reps == 0:
            continue

        joints = load_joints3d(exercise_json)
        if joints is None:
            continue

        angles = compute_angles_sequence(joints, ANGLE_TRIPLETS)
        pred_reps, _ = count_reps_adaptive(angles, exercise, ANGLE_RANGES, fps=FPS)
        acc = max(0.0, 1.0 - abs(pred_reps - gt_reps) / gt_reps)

        validation_improved.append({
            'subject':   subj,
            'exercise':  exercise,
            'gt_reps':   gt_reps,
            'pred_reps': pred_reps,
            'accuracy':  round(acc, 3),
        })

# ── Compare baseline vs improved ─────────────────────────────────────────────
accs_base = [r['accuracy'] for r in validation_results]
accs_imp  = [r['accuracy'] for r in validation_improved]

print(f'=== Baseline vs Improved Rep Counter ===')
print(f'  Sequences:         {len(accs_base)} baseline  /  {len(accs_imp)} improved')
print(f'  Mean accuracy:     {np.mean(accs_base):.3f}  →  {np.mean(accs_imp):.3f}')
print(f'  Perfect (100%):    {sum(1 for a in accs_base if a==1.0)/len(accs_base)*100:.1f}%  →  {sum(1 for a in accs_imp if a==1.0)/len(accs_imp)*100:.1f}%')
print(f'  ≥ 90% accuracy:    {sum(1 for a in accs_base if a>=0.9)/len(accs_base)*100:.1f}%  →  {sum(1 for a in accs_imp if a>=0.9)/len(accs_imp)*100:.1f}%')

from collections import defaultdict
per_ex_base = defaultdict(list)
per_ex_imp  = defaultdict(list)
for r in validation_results:   per_ex_base[r['exercise']].append(r['accuracy'])
for r in validation_improved:  per_ex_imp[r['exercise']].append(r['accuracy'])

print('\n  Per-exercise (worst improved → best improvement):')
deltas = {ex: np.mean(per_ex_imp[ex]) - np.mean(per_ex_base.get(ex, [0])) for ex in per_ex_imp}
for ex in sorted(deltas, key=deltas.get, reverse=True)[:10]:
    b = np.mean(per_ex_base.get(ex, [0]))
    i = np.mean(per_ex_imp[ex])
    print(f'    {ex:40s}  {b:.3f} → {i:.3f}  (Δ{i-b:+.3f})')


In [ ]:
# ── Save improved validation results ─────────────────────────────────────────
accs_imp  = [r['accuracy'] for r in validation_improved]

out_improved = {
    'summary': {
        'n_sequences':   len(validation_improved),
        'mean_accuracy': round(float(np.mean(accs_imp)), 4),
        'pct_perfect':   round(sum(1 for a in accs_imp if a == 1.0) / len(accs_imp) * 100, 1),
        'pct_gte90':     round(sum(1 for a in accs_imp if a >= 0.9) / len(accs_imp) * 100, 1),
        'method':        'adaptive_prominence_gaussian_smooth_median_vote',
    },
    'baseline_mean_accuracy': round(float(np.mean(accs_base)), 4),
    'results': validation_improved,
}

eval_path_imp = f'{EVAL_DIR}/rep_counter_validation_v2.json'
with open(eval_path_imp, 'w') as f:
    json.dump(out_improved, f, indent=2)

print(f'✅ Improved results saved → {eval_path_imp}')
sz = os.path.getsize(eval_path_imp) / 1024
print(f'   File size: {sz:.0f} KB')


---
## P02 Complete — Summary

| Thesis Metric | Step | Output file |
|---|---|---|
| MAE ≤ 5° angles | Step 11c | `datasets/fit3d/angle_templates/golden_angle_ranges.json` |
| Rep counter accuracy | Step 12 | `data/eval/rep_counter_validation.json` |

### What's next
- **P03**: Hook the golden angle ranges into `form_scorer.py` as `ANGLE_RANGES`
- **P04**: Run end-to-end evaluation (YOLO pose + form scorer) on the Vicon TEST set (s02, s12, s13)

### Troubleshooting
| Problem | Fix |
|---|---|
| `FileNotFoundError` on Drive path | Re-run the mount cell and verify `FIT3D_RAW` path |
| Joints array has wrong shape | Check the output of `cell-11b-sample` and adjust `JOINTS_KEY` |
| Rep count wildly off | Tune `prominence` and `distance` in `count_reps_from_angles()` |
| Joint indices look wrong | Print first frame and compare joint 0 vs joint 4/5 distance |
